# Download Common Voice 26.0 (Swahili) from Mozilla Data Collective

The archive for dataset `cmqim4c1000tmnr07zq3vwhor`
(slug `common-voice-scripted-speech-26-0-swahil-0228b2f6`) is ~20.9 GB --
**larger than a typical Kaggle session's writable quota** (commonly ~20 GB
for `/kaggle/working`, regardless of how much free space `df -h` reports on
the underlying shared host disk). `datacollective`'s `download_dataset`/
`load_dataset` always write the complete archive to disk first, so they
cannot fit here -- that's the `OSError: No space left on device` /
`DownloadError` you hit.

**This notebook never writes the full archive to disk.** It calls the MDC
API directly for a signed download URL, then streams and decompresses the
tar.gz on the fly, extracting to disk only the small files it actually
wants as it passes over them:

1. Install dependencies -> set API key -> get a signed download URL.
2. Stream through the archive once, keeping only the transcript `.tsv`
   files (a few MB).
3. Select a phoneme-balanced subset from that metadata (`select_subset.py`).
4. Stream through the archive a second time (fresh URL), keeping only the
   audio clips referenced in that subset (a few GB, not the full corpus).

Each streaming pass reads the entire compressed archive over the network
regardless of where the wanted files sit inside it (tar.gz doesn't support
seeking to a member without reading what's in front of it), so each pass
takes roughly as long as a full download -- it just doesn't need the disk
space for one.

**Before running, in this notebook's settings:** Settings -> Internet -> On.

## 1. Install dependencies

Only `python-dotenv` is needed as an extra; `requests` (used for the direct
MDC API calls) is already present in the Kaggle base image.

In [ ]:
!pip install -q python-dotenv

## 2. Set your API key as an environment variable

Generate a key at https://mozilladatacollective.com after creating an account
and agreeing to this dataset's Terms & Conditions on its MDC page.

**Recommended on Kaggle:** store it as a Kaggle Secret (Add-ons -> Secrets)
named `MDC_API_KEY` and load it at runtime as below, rather than typing the
raw key into a cell -- notebook cells and their outputs can end up saved,
shared, or version-controlled.
*If this key was ever pasted anywhere outside a secrets vault (chat, a
script, a committed file), rotate/revoke it on the MDC platform first.*

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["MDC_API_KEY"] = user_secrets.get_secret("MDC_API_KEY")
print("MDC_API_KEY loaded from Kaggle Secrets.")

## 3. Helper: get a signed download URL and stream-extract from it

Calls the same MDC API endpoint the `datacollective` SDK uses internally
(`POST /datasets/{id}/download`) to get a short-lived signed URL, without
triggering the SDK's own full-file download. `stream_extract` then reads
that URL as a streaming tar.gz (`mode="r|gz"`, forward-only, no seeking)
and writes to disk only the members for which `keep(member_name)` is True.

In [ ]:
import tarfile
from pathlib import Path

import requests

MDC_API_URL = os.environ.get("MDC_API_URL", "https://mozilladatacollective.com/api").rstrip("/")
DATASET_ID = "cmqim4c1000tmnr07zq3vwhor"  # or the slug: "common-voice-scripted-speech-26-0-swahil-0228b2f6"


def get_download_session(dataset_id: str) -> dict:
    resp = requests.post(
        f"{MDC_API_URL}/datasets/{dataset_id}/download",
        headers={"Authorization": f"Bearer {os.environ['MDC_API_KEY']}"},
        timeout=(10, 60),
    )
    resp.raise_for_status()
    return resp.json()


def stream_extract(download_url: str, keep, dest_dir: Path) -> list[str]:
    dest_dir.mkdir(parents=True, exist_ok=True)
    extracted = []
    with requests.get(download_url, stream=True, timeout=(10, 300)) as r:
        r.raise_for_status()
        with tarfile.open(fileobj=r.raw, mode="r|gz") as tar:
            for member in tar:
                if member.isfile() and keep(member.name):
                    member.name = Path(member.name).name  # flatten archive subdirectories
                    tar.extract(member, path=dest_dir)
                    extracted.append(member.name)
    return extracted

## 4. Pass 1: stream-extract just the transcript metadata

In [ ]:
session_info = get_download_session(DATASET_ID)
print("Archive size:", session_info.get("sizeBytes"))

metadata_dir = Path("/kaggle/working/cv26_sw_metadata")
extracted = stream_extract(
    session_info["downloadUrl"],
    keep=lambda name: name.endswith(".tsv"),
    dest_dir=metadata_dir,
)
print(f"Extracted {len(extracted)} TSV file(s) to {metadata_dir}")
for name in sorted(extracted):
    print(" -", name)

## 5. Select a phoneme-balanced, speaker-diverse subset

Clones the Swahili-Deepfake-dataset pipeline repo and runs
`scripts/select_subset.py` (see `docs/METHODOLOGY.md` there) to reduce
~700k+ Common Voice utterances down to a phoneme-balanced target subset --
e.g. 10,000 utterances -- before any audio is extracted.

If this repo is private for you, cloning will fail; in that case copy
`scripts/select_subset.py` and the `swahili_deepfake_dataset/` package into
this session another way (e.g. a Kaggle Dataset containing the repo)
instead of git-cloning it.

In [ ]:
!git clone --depth 1 https://github.com/regak/Swahili-Deepfake-dataset.git /kaggle/working/Swahili-Deepfake-dataset

In [ ]:
validated_tsv = metadata_dir / "validated.tsv"
if not validated_tsv.exists():
    candidates = sorted(metadata_dir.glob("*.tsv"))
    print("validated.tsv not found; available TSVs:", [p.name for p in candidates])
    if candidates:
        validated_tsv = candidates[0]
print("Using:", validated_tsv)

In [ ]:
SELECTED_SUBSET_TSV = "/kaggle/working/selected_subset.tsv"

!python /kaggle/working/Swahili-Deepfake-dataset/scripts/select_subset.py \
    "{validated_tsv}" \
    --target-size 10000 --max-per-speaker 100 \
    --output "{SELECTED_SUBSET_TSV}" \
    --report /kaggle/working/phoneme_coverage_report.json

## 6. Pass 2: stream-extract only the selected clips

Requests a fresh signed URL (the one from pass 1 may have expired by now)
and streams through the archive again, this time keeping only the audio
files referenced in `selected_subset.tsv` -- a few thousand clips instead
of the full ~700k+ corpus.

In [ ]:
import csv

with open(SELECTED_SUBSET_TSV, newline="", encoding="utf-8") as f:
    wanted = {row["id"] for row in csv.DictReader(f, delimiter="\t")}
print(f"{len(wanted)} clips selected")

clips_session_info = get_download_session(DATASET_ID)
clips_dir = Path("/kaggle/working/clips")
extracted_clips = stream_extract(
    clips_session_info["downloadUrl"],
    keep=lambda name: Path(name).name in wanted,
    dest_dir=clips_dir,
)
print(f"Extracted {len(extracted_clips)}/{len(wanted)} selected clips to {clips_dir}")

## Disk usage and persisting across sessions

Since the full archive is never written to disk, total usage stays close to
the metadata TSVs plus the selected clips (typically a few GB for a
10,000-utterance subset) -- well under a ~20 GB session quota. Check with:
```python
!du -sh /kaggle/working
```

**Nothing outside `/kaggle/working` (via Save Version) or an attached
`/kaggle/input` Dataset survives when a session ends.** To avoid re-running
both streaming passes (each ~20 GB of network transfer) in every future
session:

1. Let this notebook finish (metadata extraction, selection, clip extraction).
2. Click **Save Version** to commit the notebook and its `/kaggle/working` output.
3. From the notebook's output/Data pane, use **"New Dataset"** to publish
   that output as a private Kaggle Dataset.
4. In any future notebook, **Add Input -> your new dataset** -- it mounts
   read-only at `/kaggle/input/<dataset-slug>/` instantly, no re-streaming
   needed.